# Notebook 03 — Manual Review Validation Framework

**CS19 Capstone Project**

## Scope
This notebook constructs the **manual review validation sample** for Proposal 2 time moderation.
It is a two-stage pipeline:

1. **Draft extraction** — select 18 participant-task cases, align transcript segments, and infer
   preliminary labels from transcript evidence.
2. **Final output generation** — apply reviewer corrections, compute tie-aware MAE metrics,
   and write all deliverable files.

## Important caveats
- **Manual Review is a validation sample, not official ground truth.** The manual score is a
  reviewer-estimated anchor for comparison, not a definitive expert judgment.
- **Do not overclaim that Proposal 2 is always better.** In this 18-case sample, Model C before-time
  has the lowest MAE (13.28). Proposal 2 was clearly helpful for case_001 but noisy in several others.
- **Retry/confusion language is supporting evidence only**, never a direct scoring rule.
- **Tester Quality (Notebook 04) is an independent third score** and is not part of this validation.

## Outputs
All outputs written to `outputs/manual_review_outputs/` relative to this notebook's directory.

In [1]:
from pathlib import Path
import re, math, textwrap
import numpy as np
import pandas as pd
from datetime import datetime

BASE_DIR    = Path(".").resolve()
INPUT_DIR   = BASE_DIR
OUTPUT_DIR  = BASE_DIR / "outputs"
MAN_OUT     = OUTPUT_DIR / "manual_review_outputs"
SEGMENT_DIR = BASE_DIR / "proposal2_segments"
MAN_OUT.mkdir(parents=True, exist_ok=True)

INTEGRATED_PATH = OUTPUT_DIR / "proposal2_integrated_l3_time_score.csv"
SEGMAP_PATH     = INPUT_DIR  / "task_segmentation_combined.csv"

for p in [INTEGRATED_PATH, SEGMAP_PATH, SEGMENT_DIR]:
    assert p.exists(), f"Missing: {p}"

print("BASE_DIR :", BASE_DIR)
print("MAN_OUT  :", MAN_OUT)
print("Segments :", len(list(SEGMENT_DIR.glob('*_segments.csv'))), "files")

BASE_DIR : /home/ubuntu/cs19_capstone
MAN_OUT  : /home/ubuntu/cs19_capstone/outputs/manual_review_outputs
Segments : 21 files


## Stage 1 — Draft case extraction

### Selected cases (18 participant-task pairs)
Cases were chosen to cover: changed-score examples, cap-triggered-but-unchanged, high normalized
time ratios, medium-time/no-cap, normal-time controls, and missing-evidence cases.

In [2]:
SELECTED = [
    ('ghum@me.com',                   'AAMI Website',                                                    'changed_score'),
    ('ramazankawish@gmail.com',        'AAMI Website',                                                    'changed_score'),
    ('kris.rae@live.com.au',           'PDFs and Understanding Terms',                                    'changed_score'),
    ('sharelinsonny@gmail.com',        'Equal Access to Critical  Information – Emergency Response Page', 'changed_score'),
    ('sharelinsonny@gmail.com',        'Finding Answers',                                                 'changed_score'),
    ('gameoverdan@gmail.com',          'Finding Answers',                                                 'cap_triggered_not_changed'),
    ('fjone7@gmail.com',               'AAMI Website',                                                    'medium_time_no_cap'),
    ('reneerussell99@hotmail.com',     'AAMI Website',                                                    'medium_time_no_cap'),
    ('carlpatrickrobinson@gmail.com',  'AAMI Website',                                                    'control_normal_time'),
    ('gracieha22@gmail.com',           'AAMI Website',                                                    'control_normal_time'),
    ('gameoverdan@gmail.com',          'AAMI Website',                                                    'control_normal_time_low_score'),
    ('oliviamitchell22@gmail.com',     'Equal Access to Critical  Information – Emergency Response Page', 'control_normal_time'),
    ('mgblackwell2001@gmail.com',      'PDFs and Understanding Terms',                                    'medium_time_no_cap'),
    ('margieflint@iprimus.com.au',     'PDFs and Understanding Terms',                                    'mild_time_no_cap'),
    ('reneerussell99@hotmail.com',     'PDFs and Understanding Terms',                                    'control_normal_time'),
    ('reececraigie@outlook.com',       'AAMI Website',                                                    'missing_time_evidence'),
    ('thanoptions@icloud.com',         'AAMI Website',                                                    'missing_time_evidence'),
    ('terryaflint17@gmail.com',        'Finding Answers',                                                 'missing_time_evidence'),
]

PREFERRED_QUESTION = {
    'AAMI Website': [
        'Overall, how accessible did you find the AAMI website?',
        'How easy was it to navigate around the website?',
        'Generally speaking',
    ],
    'PDFs and Understanding Terms': ['How easy was it to find and read a product disclosure statement?'],
    'Equal Access to Critical  Information – Emergency Response Page': ['How easy and clear was the information on Natural disasters?'],
    'Finding Answers': ['How user-friendly was the search function?', 'How user friendly was the live chat?'],
}

MODEL_TASK_TO_TASK_ID = {
    'AAMI Website': 2,
    'PDFs and Understanding Terms': 5,
    'Equal Access to Critical  Information – Emergency Response Page': 7,
    'Finding Answers': 10,
}

print(f"Selected cases: {len(SELECTED)}")

Selected cases: 18


In [3]:
# ── Helpers ────────────────────────────────────────────────────────────────
def email_local(email):
    return re.sub(r'[^a-z0-9]', '', str(email).split('@')[0].lower())

def norm_name(s):
    if pd.isna(s): return ''
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def hms_to_seconds(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)): return float(x)
    s = str(x).strip()
    if re.fullmatch(r'\d+(\.\d+)?', s): return float(s)
    parts = s.split(':')
    try: parts = [float(p) for p in parts]
    except Exception: return np.nan
    if len(parts)==3: return parts[0]*3600+parts[1]*60+parts[2]
    if len(parts)==2: return parts[0]*60+parts[1]
    if len(parts)==1: return parts[0]
    return np.nan

def seconds_to_hms(x):
    if x is None or pd.isna(x): return ''
    x = float(x); h=int(x//3600); m=int((x%3600)//60); s=int(round(x%60))
    return f'{h:02d}:{m:02d}:{s:02d}'

FRICTION_TERMS = [
    'difficult','difficulty','hard','confusing','confused','not sure','unsure',
    'cannot','can\'t','couldn\'t','could not','unable','stuck','problem','issue',
    'trouble','frustrat','where is','where do','i don\'t know','do not know',
    'not obvious','not clear','took me','take a while','struggle','lost',
    'give up','not find','couldn\'t find','can\'t find','cannot find',
    'try again','back','search again','annoying'
]
POSITIVE_TERMS = [
    'easy','straightforward','found','i found','there it is','clear','helpful',
    'good','simple','quick','intuitive','pretty good','works','able to',
    'success','perfect','makes sense'
]
COMPLETION_TERMS = [
    'found','there it is','i can see','i found','able to','got it',
    'here we go','success','that answers','this answers','click','opened',
    'it shows','i would'
]

def score_terms(text, terms):
    tl = text.lower()
    return sum(tl.count(t.lower()) for t in terms)

def choose_excerpt(segdf, start, end, max_words=230):
    subset = segdf[((segdf['end_time'] >= start-2) & (segdf['start_time'] <= end+2))].copy()
    if subset.empty:
        return '', subset
    subset['score'] = subset['text'].fillna('').apply(
        lambda t: score_terms(t, FRICTION_TERMS)*3 + score_terms(t, POSITIVE_TERMS) + score_terms(t, COMPLETION_TERMS))
    all_words = ' '.join(subset['text'].fillna('').astype(str)).split()
    if len(all_words) <= max_words:
        return ' '.join(all_words), subset
    top_idx = list(subset.sort_values('score', ascending=False).head(3).index)
    selected_idx = []
    if len(subset)>0: selected_idx.append(subset.index[0])
    selected_idx += top_idx
    if len(subset)>1: selected_idx.append(subset.index[-1])
    selected_idx = sorted(set(selected_idx))
    txt = ' '.join(subset.loc[selected_idx, 'text'].fillna('').astype(str))
    words = txt.split()
    if len(words) > max_words:
        txt = ' '.join(words[:max_words]) + ' ...'
    return txt, subset

def infer_label(text, ratio, selected_reason, has_overlap, task_conf, has_missing_core):
    fl  = score_terms(text, FRICTION_TERMS)
    po  = score_terms(text, POSITIVE_TERMS)
    comp = score_terms(text, COMPLETION_TERMS)
    tl  = text.lower()
    severe_any = any(x in tl for x in ["give up","could not find","couldn't find",
                                        "cannot find","can't find","unable","stuck",
                                        "not accessible","not working"])
    if has_missing_core or not has_overlap or len(text.split()) < 30:
        return 'insufficient_evidence', np.nan, 'low'
    if severe_any and fl >= 2:
        return 'severe_friction_or_failure', 25, 'high' if task_conf=='high' else 'medium'
    if fl >= 5 or (pd.notna(ratio) and ratio >= 3 and fl >= 2):
        return 'moderate_friction', 55, 'medium'
    if fl >= 2 or (pd.notna(ratio) and ratio >= 2 and selected_reason.startswith('changed')):
        label = 'minor_friction' if po >= comp and fl < 4 else 'moderate_friction'
        score = 75 if (po >= comp and fl < 4) else 55
        return label, score, 'medium'
    if comp >= 1 or po >= 2:
        return 'smooth_success', 90, 'high' if task_conf=='high' else 'medium'
    return 'insufficient_evidence', np.nan, 'low'

def evidence_sentences(text, terms, max_items=3):
    if not text: return ''
    sents = re.split(r'(?<=[.!?])\s+|\s{2,}', text)
    hits = []
    for sent in sents:
        if any(t.lower().replace("\\'","'") in sent.lower() for t in terms):
            hits.append(sent.strip())
        if len(hits) >= max_items: break
    return ' | '.join(hits)

print("Helper functions defined.")

Helper functions defined.


In [4]:
# ── Load data ──────────────────────────────────────────────────────────────
integrated = pd.read_csv(INTEGRATED_PATH)
segmap     = pd.read_csv(SEGMAP_PATH)

# Build segment file index
seg_files = list(SEGMENT_DIR.glob('*_segments.csv'))
seg_file_by_norm = {norm_name(p.name.replace('_segments.csv','')): p for p in seg_files}

print(f"Integrated rows: {len(integrated)}")
print(f"Segment files  : {len(seg_files)}")

Integrated rows: 136
Segment files  : 21


In [5]:
# ── Build draft cases ──────────────────────────────────────────────────────
case_rows = []

for idx, (email, mtask, reason) in enumerate(SELECTED, start=1):
    g = integrated[(integrated['email'].str.lower()==email.lower()) &
                   (integrated['model_c_task']==mtask)].copy()
    if g.empty:
        raise ValueError(f"Not found in integrated: ({email}, {mtask})")

    pref = PREFERRED_QUESTION.get(mtask, [])
    def qrank(q):
        for i, p in enumerate(pref):
            if p.lower() in str(q).lower(): return i
        return 99
    g['_qrank']   = g['survey_question'].map(qrank)
    g['_changed'] = g['proposal2_changed_score'].astype(str).str.lower().eq('true')
    g = g.sort_values(['_changed','_qrank','l3_score_before_time'], ascending=[False,True,False])
    row = g.iloc[0]

    # Resolve participant name
    participant = str(row.get('participant','')) if pd.notna(row.get('participant','')) else ''
    if not participant:
        local = email_local(email)
        participant = seg_file_by_norm.get(local, {})
        participant = local
    pnorm = norm_name(participant)

    # Find segment file
    segfile = seg_file_by_norm.get(pnorm) or seg_file_by_norm.get(email_local(email))

    # Task timing from segmap
    task_id = row['task_id'] if pd.notna(row.get('task_id')) else MODEL_TASK_TO_TASK_ID.get(mtask)
    segrow = pd.DataFrame()
    if task_id is not None and not pd.isna(task_id):
        segrow = segmap[
            (segmap['participant'].map(norm_name)==norm_name(participant)) &
            (segmap['task_id'].astype(str).str.replace('.0','',regex=False)==str(int(float(task_id))))
        ]
    if segrow.empty and task_id is not None:
        segrow = segmap[
            (segmap['participant'].map(norm_name)==email_local(email)) &
            (segmap['task_id'].astype(str).str.replace('.0','',regex=False)==str(int(float(task_id))))
        ]

    task_start = task_end = np.nan; task_conf = ''; evidence_phrase = ''
    if not segrow.empty:
        sr = segrow.iloc[0]
        task_start = hms_to_seconds(sr['approximate_start_time'])
        task_end   = hms_to_seconds(sr['approximate_end_time'])
        task_conf  = str(sr.get('confidence',''))
        evidence_phrase = str(sr.get('evidence_phrase',''))

    # Extract transcript
    transcript_excerpt = ''; subset = pd.DataFrame(); extraction_note = ''
    if segfile and segfile.exists() and pd.notna(task_start) and pd.notna(task_end):
        segdf = pd.read_csv(segfile)
        if 'start_seconds' in segdf and 'start_time' not in segdf:
            segdf['start_time'] = segdf['start_seconds']
        if 'end_seconds' in segdf and 'end_time' not in segdf:
            segdf['end_time'] = segdf['end_seconds']
        if 'text' not in segdf:
            txt_cols = [c for c in segdf.columns if 'text' in c.lower() or 'transcript' in c.lower()]
            segdf['text'] = segdf[txt_cols[0]] if txt_cols else ''
        segdf['start_time'] = pd.to_numeric(segdf['start_time'], errors='coerce')
        segdf['end_time']   = pd.to_numeric(segdf['end_time'],   errors='coerce')
        transcript_excerpt, subset = choose_excerpt(segdf, task_start, task_end)
    else:
        extraction_note = 'Missing segment file or task time range; could not align transcript evidence.'

    has_overlap      = not subset.empty
    has_missing_core = ('missing_time_evidence' in reason and
                        (pd.isna(row.get('task_id')) or pd.isna(row.get('time_signal'))))

    manual_label, manual_score, manual_conf = infer_label(
        transcript_excerpt, row.get('normalized_median_ratio'), reason,
        has_overlap, task_conf, has_missing_core)

    fl   = score_terms(transcript_excerpt, FRICTION_TERMS)
    po   = score_terms(transcript_excerpt, POSITIVE_TERMS)
    comp = score_terms(transcript_excerpt, COMPLETION_TERMS)
    friction_ev  = evidence_sentences(transcript_excerpt, FRICTION_TERMS) or (
        'No clear retry/confusion evidence.' if manual_label!='insufficient_evidence'
        else 'Unclear due to transcript noise or missing aligned task evidence.')
    positive_ev  = evidence_sentences(transcript_excerpt, COMPLETION_TERMS+POSITIVE_TERMS) or (
        'No clear completion evidence in the aligned excerpt.' if manual_label!='insufficient_evidence'
        else 'Completion could not be confidently assessed from aligned transcript evidence.')

    if manual_label=='smooth_success':
        summary = f'Transcript shows low explicit friction and some progress/completion language; time signal is {row.get("time_signal")}.'
    elif manual_label=='minor_friction':
        ratio_val = row.get('normalized_median_ratio')
        ratio_str = f'{ratio_val:.2f}' if pd.notna(ratio_val) else 'N/A'
        summary = f'Transcript indicates limited hesitation or effort but not clear failure; normalized time ratio={ratio_str} where available.'
    elif manual_label=='moderate_friction':
        summary = 'Transcript and/or high normalized duration indicate repeated effort or uncertainty, but evidence does not prove complete failure.'
    elif manual_label=='severe_friction_or_failure':
        summary = 'Transcript contains explicit inability, getting stuck, or task failure evidence.'
    else:
        summary = 'Aligned transcript evidence or Proposal 2 task mapping is insufficient for a confident manual judgement.'

    reviewer_note_parts = []
    if extraction_note: reviewer_note_parts.append(extraction_note)
    if task_conf and task_conf.lower()!='high': reviewer_note_parts.append(f'Task segmentation confidence={task_conf}.')
    if has_missing_core: reviewer_note_parts.append('Integrated Proposal 2 row has missing participant/task/time evidence; retained as insufficient-evidence validation case.')
    if segfile is None: reviewer_note_parts.append('No matching segment file found by participant/email-local name.')

    survey_answer = ''
    if pd.notna(row.get('l1_score')):
        survey_answer = f"{row.get('l1_score')/20:g} / 5 Likert-equivalent"

    case_rows.append({
        'case_id':                    f'case_{idx:03d}',
        'selection_reason':           reason,
        'participant':                participant,
        'email':                      email,
        'task_id':                    '' if pd.isna(task_id) else int(float(task_id)),
        'task_title':                 row.get('task_title') if pd.notna(row.get('task_title')) else mtask,
        'model_c_task':               mtask,
        'segment_file':               segfile.name if segfile else '',
        'task_start_time':            seconds_to_hms(task_start),
        'task_end_time':              seconds_to_hms(task_end),
        'transcript_excerpt':         transcript_excerpt,
        'survey_question':            row.get('survey_question'),
        'survey_answer':              survey_answer,
        'raw_survey_score':           row.get('l1_score'),
        'l3_score_before_time':       row.get('l3_score_before_time'),
        'proposal2_time_cap':         row.get('proposal2_time_cap'),
        'l3_score_after_proposal2_time': row.get('l3_score_after_proposal2_time'),
        'normalized_median_ratio':    row.get('normalized_median_ratio'),
        'time_signal':                row.get('time_signal'),
        'manual_label':               manual_label,
        'manual_ground_truth_score':  manual_score,
        'manual_confidence':          manual_conf,
        'manual_evidence_summary':    summary,
        'retry_or_confusion_evidence': friction_ev,
        'completion_evidence':        positive_ev,
        'reviewer_note':              ' '.join(reviewer_note_parts),
        '_friction_count':            fl,
        '_positive_count':            po,
        '_completion_count':          comp,
        '_task_seg_confidence':       task_conf,
        '_segments_used':             0 if subset.empty else len(subset),
    })

draft = pd.DataFrame(case_rows)
draft.to_csv(MAN_OUT / 'manual_review_cases_draft.csv', index=False)
print(f"Draft cases written: {len(draft)} rows")
print(draft[['case_id','participant','model_c_task','selection_reason','manual_label',
             'manual_ground_truth_score','_segments_used']].to_string(index=False))

Draft cases written: 18 rows
 case_id         participant                                                    model_c_task              selection_reason               manual_label  manual_ground_truth_score  _segments_used
case_001                ghum                                                    AAMI Website                 changed_score          moderate_friction                       55.0              41
case_002       ramazankawish                                                    AAMI Website                 changed_score severe_friction_or_failure                       25.0             105
case_003                 rae                                    PDFs and Understanding Terms                 changed_score          moderate_friction                       55.0              86
case_004       sharelinsonny Equal Access to Critical  Information – Emergency Response Page                 changed_score          moderate_friction                       55.0             109
case_0

## Stage 2 — Reviewer corrections and final output generation

The corrections below were determined by the reviewer after inspecting task-aligned transcript
windows. They overwrite the draft inferred labels where the automated inference was incorrect.
Retry/repetition language is used only as supporting evidence, never as a direct scoring rule.

In [6]:
import numpy as np  # ensure np.nan available in corrections dict

CORRECTIONS = {
    'case_001': dict(
        manual_label='moderate_friction', manual_ground_truth_score=55, manual_confidence='medium',
        manual_evidence_summary='Task evidence shows real navigation and readability friction, including difficulty navigating headings and interpreting dense homepage content, but not a clear total task failure.',
        retry_or_confusion_evidence='Explicit confusion/friction includes "I can\'t navigate through the various kinds of insurances by heading" and difficulty deciding how much stock to put in homepage reviews.',
        completion_evidence='The participant reached AAMI content and explored insurance options, but the experience remained effortful and only partially smooth.',
        reviewer_note='Selected changed-score / strong-time-friction case. Screen-reader and page text are mixed, so repeated headings were not treated as direct retry evidence.'
    ),
    'case_002': dict(
        manual_label='smooth_success', manual_ground_truth_score=90, manual_confidence='high',
        transcript_excerpt='The participant repeatedly described the AAMI website as easy to understand and accessible: "The products are here which is a good thing… new [visitors] can find everything in here." They found product categories, discounts, contact pathways and phone information, then summarized: "Bottom line the website is very accessible, very useful, trustworthy. Everything is clear… I can find everything easily."',
        manual_evidence_summary='Despite a long duration and some exploratory browsing, the participant consistently reported that the homepage and product information were clear, accessible, and easy to find.',
        retry_or_confusion_evidence='Minor uncertainty appeared around finding contact details and whether the right page was opened, but it was resolved and did not prevent task success.',
        completion_evidence='The participant found product categories, discounts, contact options, and stated that the website was accessible and useful.',
        reviewer_note='Time signal appears noisy because the participant gave an extended verbal evaluation rather than struggling with the task.'
    ),
    'case_003': dict(
        manual_label='moderate_friction', manual_ground_truth_score=55, manual_confidence='high',
        transcript_excerpt='For the product disclosure statement, the participant found the PDF but reacted to its size: "104 pages, oh my goodness… that\'s a bit much." On mobile, they said they had to zoom in and that touching near the edge made the PDF scroll unexpectedly, which was "very frustrating." They still noted helpful structure, headings, dot points, glossary, page breaks, and suggested an easy-read version and better hyperlinks for accessibility.',
        manual_evidence_summary='The participant completed the PDF task but experienced substantial friction from document length, mobile zoom/scroll behaviour, and accessibility needs.',
        retry_or_confusion_evidence='Clear friction includes "104 pages… that\'s a bit much," "I have to zoom in," and accidental scrolling that was "very frustrating."',
        completion_evidence='The participant opened the PDF, reviewed its structure and terms, and provided specific evaluation comments.',
        reviewer_note='The task window also contains later task material; the manual label uses only the product-disclosure portion of the transcript.'
    ),
    'case_004': dict(
        manual_label='minor_friction', manual_ground_truth_score=75, manual_confidence='medium',
        transcript_excerpt='In the emergency/weather-information task, the participant located support-oriented information and described the service as useful during stressful claims situations. They commented that headings and sections helped them jump to relevant parts, while also noting some uncertainty and account/document access concerns in adjacent material.',
        manual_evidence_summary='The participant found relevant emergency/support information and evaluated it positively, with only limited uncertainty in the aligned task evidence.',
        retry_or_confusion_evidence='Some uncertainty appears in nearby transcript material, but the task-aligned emergency evidence does not prove major failure.',
        completion_evidence='The participant identified weather/emergency support content and described it as conveying value during stressful claim situations.',
        reviewer_note='Segmentation confidence is medium and adjacent-task speech is mixed in; therefore this was reviewed conservatively as minor friction rather than strong time friction.'
    ),
    'case_005': dict(
        manual_label='minor_friction', manual_ground_truth_score=75, manual_confidence='high',
        transcript_excerpt='For finding answers, the participant looked for missed-payment and support information, then found payment-management instructions: "your policy won\'t be canceled right away," "you can manage your regular payment in just a few minutes," and contact options by phone/email. They later tried menus/search/live chat; although they said "I don\'t find what I\'m trying to find" briefly, they used chat and concluded that the website let customers solve questions themselves "without unnecessary frustration."',
        manual_evidence_summary='The task involved several answer-finding routes and a brief failed menu/search attempt, but the participant found actionable information and support channels.',
        retry_or_confusion_evidence='Explicit minor confusion: "I don\'t find what I\'m trying to find," followed by switching to live chat.',
        completion_evidence='The participant found missed-payment guidance, online payment steps, contact details, and chat support, then concluded the site supported self-service answers.',
        reviewer_note='Strong time ratio is noisy here because the participant repeated/extended a multi-route evaluation rather than failing the task.'
    ),
    'case_006': dict(
        manual_label='severe_friction_or_failure', manual_ground_truth_score=25, manual_confidence='high',
        manual_evidence_summary='The participant explicitly failed to find the search function and described the task as a fail/no-go.',
        retry_or_confusion_evidence='Strong explicit evidence: "I truly can\'t find it," "that\'s a fail on that one," and "I cannot find a search function whatsoever."',
        completion_evidence='No clear completion; the participant gave up on finding the search function.',
        reviewer_note='Cap-triggered but unchanged case; low pre-time score already captured the severe experience.'
    ),
    'case_007': dict(
        manual_label='smooth_success', manual_ground_truth_score=90, manual_confidence='medium',
        manual_evidence_summary='The participant gave an extended qualitative review, but the task evidence is broadly positive and does not indicate substantive failure.',
        retry_or_confusion_evidence='No clear retry/confusion evidence. The long duration appears to reflect verbose commentary and screen-reader/page exploration.',
        completion_evidence='The participant explored the homepage, product categories, and community/history content and described aspects as accessible and working well.',
        reviewer_note='Medium-time/no-cap case. Duration looks less diagnostic than transcript content.'
    ),
    'case_008': dict(
        manual_label='smooth_success', manual_ground_truth_score=90, manual_confidence='high',
        manual_evidence_summary='The participant explicitly reported no accessibility, screen-reader, or confusing-information issues on the homepage.',
        retry_or_confusion_evidence='No clear retry/confusion evidence; only mild uncertainty about carousel-style scroll buttons.',
        completion_evidence='The participant browsed the homepage from top to bottom and stated that it was laid out well and not convoluted.',
        reviewer_note='Medium-time/no-cap case where transcript evidence suggests the time signal is not meaningful friction.'
    ),
    'case_009': dict(
        manual_label='minor_friction', manual_ground_truth_score=75, manual_confidence='medium',
        manual_evidence_summary='The participant needed some time to understand "contents insurance" but then found and understood the relevant information.',
        retry_or_confusion_evidence='Minor hesitation: "It took me a while to figure out what contents insurance was," followed by understanding.',
        completion_evidence='The participant reached home and contents information and said the website presented it clearly.',
        reviewer_note='Normal-time control case.'
    ),
    'case_010': dict(
        manual_label='smooth_success', manual_ground_truth_score=90, manual_confidence='high',
        manual_evidence_summary='The participant found navigation simple and identified relevant homepage shortcuts and insurance categories quickly.',
        retry_or_confusion_evidence='No clear retry/confusion evidence.',
        completion_evidence='The participant stated navigation was simple and identified home insurance, home contents, and building-only options.',
        reviewer_note='Normal-time control case.'
    ),
    'case_011': dict(
        manual_label='severe_friction_or_failure', manual_ground_truth_score=25, manual_confidence='high',
        manual_evidence_summary='The participant found the site only with effort and then reported substantial accessibility barriers caused by small text, visual complexity, inversion, and repeated zoom/scrolling.',
        retry_or_confusion_evidence='Clear accessibility-friction evidence: "hard to navigate," "hard to see," "writing itself is terribly small," and needing to scroll "back and forward… side to side."',
        completion_evidence='The participant reached the homepage, but the experience contained major accessibility friction rather than smooth task success.',
        reviewer_note='Normal-time low-score control case; transcript supports the low model/survey score despite normal duration.'
    ),
    'case_012': dict(
        manual_label='insufficient_evidence', manual_ground_truth_score=np.nan, manual_confidence='low',
        manual_evidence_summary='No transcript segments overlapped the selected task time window, so the case cannot support a confident manual judgment.',
        retry_or_confusion_evidence='Unclear due to missing aligned transcript evidence.',
        completion_evidence='Completion could not be assessed from the available aligned transcript.',
        reviewer_note='No overlapping transcript segments for this task window; excluded from error metrics.'
    ),
    'case_013': dict(
        manual_label='minor_friction', manual_ground_truth_score=75, manual_confidence='medium',
        manual_evidence_summary='The participant found useful PDF/terms information but described the route as somewhat annoying and a little confusing.',
        retry_or_confusion_evidence='Minor friction evidence includes "somewhat annoying" and "a little bit confusing to navigate to."',
        completion_evidence='The participant identified helpful index/term information and understood what was included or excluded.',
        reviewer_note='Medium-time/no-cap PDF case; transcript suggests friction was present but not severe.'
    ),
    'case_014': dict(
        manual_label='minor_friction', manual_ground_truth_score=75, manual_confidence='high',
        transcript_excerpt='For task 5, the participant said: "we\'re looking at the product disclosure statement… Home and property… Get documents." They found the relevant product disclosure statement further down the page: "there we go… that didn\'t take very long," then opened a 103-page PDF. They described it as extensive and detailed, noted that key responsibilities and exclusions were important, and suggested the site should make the need to read the PDF more prominent on the main page.',
        manual_evidence_summary='The participant found the PDS without major difficulty and understood its content, but the length/detail of the PDF and prominence of key responsibilities created minor friction.',
        retry_or_confusion_evidence='Minor friction includes uncertainty about the search button/get-documents route and concern that PDF responsibilities should be highlighted more clearly.',
        completion_evidence='The participant found and reviewed the product disclosure statement, including coverage limits, exclusions, and responsibilities.',
        reviewer_note='Draft severe label was corrected after reviewing the actual task-5 segment; the transcript is mostly successful and positive.'
    ),
    'case_015': dict(
        manual_label='minor_friction', manual_ground_truth_score=75, manual_confidence='high',
        transcript_excerpt='The participant looked for the home/building insurance disclosure statement, found "product disclosure statement," downloaded it, and reviewed the table of contents, blank-page note, glossary, and broad structure. They concluded that the document "looks easy to read and understand," and that although it was "buried quite deep in the navigation," the route "made sense" and they were "still able to find it and know exactly where to go."',
        manual_evidence_summary='The participant completed the PDF task and understood the document, with minor friction because the PDS was relatively deep in navigation.',
        retry_or_confusion_evidence='Minor friction: "buried quite deep in the navigation," but resolved successfully.',
        completion_evidence='The participant found and downloaded the PDS, reviewed its contents and glossary, and stated they knew where to find it.',
        reviewer_note='Draft severe label was corrected; transcript evidence shows task success with minor navigation depth friction.'
    ),
    'case_016': dict(
        manual_label='insufficient_evidence', manual_ground_truth_score=np.nan, manual_confidence='low',
        manual_evidence_summary='Missing time evidence or segment alignment prevents a confident manual judgment.',
        retry_or_confusion_evidence='Unclear due to missing aligned transcript evidence.',
        completion_evidence='Completion could not be assessed from the available aligned transcript.',
        reviewer_note='Integrated Proposal 2 row has missing participant/task/time evidence; retained as insufficient-evidence validation case and excluded from error metrics.'
    ),
    'case_017': dict(
        manual_label='insufficient_evidence', manual_ground_truth_score=np.nan, manual_confidence='low',
        manual_evidence_summary='Missing time evidence or segment alignment prevents a confident manual judgment.',
        retry_or_confusion_evidence='Unclear due to missing aligned transcript evidence.',
        completion_evidence='Completion could not be assessed from the available aligned transcript.',
        reviewer_note='Integrated Proposal 2 row has missing participant/task/time evidence; retained as insufficient-evidence validation case and excluded from error metrics.'
    ),
    'case_018': dict(
        manual_label='insufficient_evidence', manual_ground_truth_score=np.nan, manual_confidence='low',
        manual_evidence_summary='Missing time evidence or segment alignment prevents a confident manual judgment.',
        retry_or_confusion_evidence='Unclear due to missing aligned transcript evidence.',
        completion_evidence='Completion could not be assessed from the available aligned transcript.',
        reviewer_note='Integrated Proposal 2 row has missing participant/task/time evidence; retained as insufficient-evidence validation case and excluded from error metrics.'
    ),
}
print(f"Corrections defined for {len(CORRECTIONS)} cases.")

Corrections defined for 18 cases.


In [7]:
# ── Apply corrections ──────────────────────────────────────────────────────
df = draft.copy()
for idx, row in df.iterrows():
    corr = CORRECTIONS.get(row['case_id'], {})
    for k, v in corr.items():
        df.at[idx, k] = v

# Clean and shorten text fields
def clean_text(x):
    if pd.isna(x): return ''
    return ' '.join(str(x).replace('\n',' ').split())

def shorten_words(text, max_words=230):
    words = clean_text(text).split()
    if len(words) <= max_words: return ' '.join(words)
    return ' '.join(words[:max_words]) + ' ...'

for i in df.index:
    df.at[i,'transcript_excerpt'] = shorten_words(df.at[i,'transcript_excerpt'], 230)
    for c in ['manual_evidence_summary','retry_or_confusion_evidence','completion_evidence','reviewer_note']:
        df.at[i, c] = clean_text(df.at[i, c])

# Final required columns
required_cols = [
    'case_id','participant','email','task_id','task_title','model_c_task','segment_file',
    'task_start_time','task_end_time','transcript_excerpt','survey_question','survey_answer',
    'raw_survey_score','l3_score_before_time','proposal2_time_cap','l3_score_after_proposal2_time',
    'normalized_median_ratio','time_signal','manual_label','manual_ground_truth_score',
    'manual_confidence','manual_evidence_summary','retry_or_confusion_evidence',
    'completion_evidence','reviewer_note'
]
for c in required_cols:
    if c not in df.columns: df[c] = np.nan
cases = df[required_cols].copy()
cases.to_csv(MAN_OUT / 'manual_review_cases.csv', index=False)
print(f"manual_review_cases.csv written: {len(cases)} rows × {len(cases.columns)} cols")
print("Label distribution:")
print(cases['manual_label'].value_counts(dropna=False).to_string())

manual_review_cases.csv written: 18 rows × 25 cols
Label distribution:
manual_label
minor_friction                6
smooth_success                4
insufficient_evidence         4
moderate_friction             2
severe_friction_or_failure    2


In [8]:
# ── Rubric ─────────────────────────────────────────────────────────────────
rubric = pd.DataFrame([
    dict(manual_label='smooth_success',          manual_ground_truth_score=90,
         evidence_criteria='Clear task success, information found, no obvious confusion, smooth navigation, confidence in answer.',
         validation_use='Included in error metrics.'),
    dict(manual_label='minor_friction',          manual_ground_truth_score=75,
         evidence_criteria='Small hesitation, slight search effort, minor confusion resolved quickly, task still completed without major issue.',
         validation_use='Included in error metrics.'),
    dict(manual_label='moderate_friction',       manual_ground_truth_score=55,
         evidence_criteria='Repeated searching, unclear navigation, uncertainty, long effort, partial success, or workaround; not necessarily total failure.',
         validation_use='Included in error metrics.'),
    dict(manual_label='severe_friction_or_failure', manual_ground_truth_score=25,
         evidence_criteria='Could not complete task, gave up, got stuck, could not find key information, clear accessibility barrier, or major workaround.',
         validation_use='Included in error metrics.'),
    dict(manual_label='insufficient_evidence',   manual_ground_truth_score=np.nan,
         evidence_criteria='Transcript incomplete, task boundary unclear, speech mixed with screen-reader/page audio, or no aligned task evidence.',
         validation_use='Excluded from error metrics; retained for auditability.'),
])
rubric.to_csv(MAN_OUT / 'manual_review_scoring_rubric.csv', index=False)
print("manual_review_scoring_rubric.csv written.")

manual_review_scoring_rubric.csv written.


In [9]:
# ── Validation metrics ─────────────────────────────────────────────────────
metric_df = cases.copy()
for col in ['raw_survey_score','l3_score_before_time','l3_score_after_proposal2_time','manual_ground_truth_score']:
    metric_df[col] = pd.to_numeric(metric_df[col], errors='coerce')
valid = metric_df.dropna(subset=['manual_ground_truth_score']).copy()
valid['error_raw_survey_score']    = (valid['raw_survey_score']    - valid['manual_ground_truth_score']).abs()
valid['error_model_c_before_time'] = (valid['l3_score_before_time']- valid['manual_ground_truth_score']).abs()
valid['error_proposal2_after_time']= (valid['l3_score_after_proposal2_time'] - valid['manual_ground_truth_score']).abs()

metric_map = {
    'raw_survey_score':    'error_raw_survey_score',
    'model_c_before_time': 'error_model_c_before_time',
    'proposal2_after_time':'error_proposal2_after_time',
}

closest_counts = {m: 0 for m in metric_map}
unique_counts  = {m: 0 for m in metric_map}
tie_rows = 0
for _, r in valid.iterrows():
    errs = {m: r[e] for m, e in metric_map.items()}
    min_err = min(errs.values())
    closest = [m for m, e in errs.items() if abs(e - min_err) < 1e-9]
    for m in closest: closest_counts[m] += 1
    if len(closest) == 1: unique_counts[closest[0]] += 1
    else: tie_rows += 1

summary_rows = []
for m, err_col in metric_map.items():
    summary_rows.append(dict(
        metric=m,
        mean_absolute_error=round(float(valid[err_col].mean()), 2),
        median_absolute_error=round(float(valid[err_col].median()), 2),
        n_cases=int(valid[err_col].notna().sum()),
        closest_count_including_ties=int(closest_counts[m]),
        closest_percentage_including_ties=round(100*closest_counts[m]/len(valid),1) if len(valid) else np.nan,
        unique_closest_count=int(unique_counts[m]),
        unique_closest_percentage=round(100*unique_counts[m]/len(valid),1) if len(valid) else np.nan,
        notes='Closest counts are tie-aware; percentages can sum above 100 when several scores tie.'
    ))
summary_rows.append(dict(
    metric='tie_cases_multiple_scores_closest',
    mean_absolute_error=np.nan, median_absolute_error=np.nan,
    n_cases=int(len(valid)),
    closest_count_including_ties=int(tie_rows),
    closest_percentage_including_ties=round(100*tie_rows/len(valid),1) if len(valid) else np.nan,
    unique_closest_count=np.nan, unique_closest_percentage=np.nan,
    notes='Number of valid cases where two or three candidate scores were equally close to the manual score.'
))
summary = pd.DataFrame(summary_rows)
summary.to_csv(MAN_OUT / 'manual_review_validation_summary.csv', index=False)
print("Validation summary:")
print(summary[['metric','mean_absolute_error','n_cases']].to_string(index=False))

Validation summary:
                           metric  mean_absolute_error  n_cases
                 raw_survey_score                13.57       14
              model_c_before_time                13.28       14
             proposal2_after_time                19.49       14
tie_cases_multiple_scores_closest                  NaN       14


In [10]:
# ── Error details ──────────────────────────────────────────────────────────
error_detail_cols = [
    'case_id','participant','task_id','model_c_task','manual_label',
    'manual_ground_truth_score','raw_survey_score','l3_score_before_time',
    'l3_score_after_proposal2_time','error_raw_survey_score',
    'error_model_c_before_time','error_proposal2_after_time'
]
valid[error_detail_cols].to_csv(MAN_OUT / 'manual_review_error_details.csv', index=False)
print("manual_review_error_details.csv written.")

manual_review_error_details.csv written.


In [11]:
exec(open(BASE_DIR / 'nb03_findings_cell.py').read())

manual_review_findings.md written.


In [12]:
exec(open(BASE_DIR / 'nb03_examples_cell.py').read())

manual_review_case_examples.md written.


In [13]:
# ── Quality check ─────────────────────────────────────────────────────────
qc_lines = ['# Manual Review Output Quality Check']
def check(label, condition, detail=''):
    status = 'PASS' if condition else 'FAIL'
    qc_lines.append(f'{status} {label}' + (f': {detail}' if detail else ''))
    return condition

check('file_exists::manual_review_cases.csv',            (MAN_OUT/'manual_review_cases.csv').exists())
check('file_exists::manual_review_scoring_rubric.csv',   (MAN_OUT/'manual_review_scoring_rubric.csv').exists())
check('file_exists::manual_review_validation_summary.csv',(MAN_OUT/'manual_review_validation_summary.csv').exists())
check('file_exists::manual_review_findings.md',          (MAN_OUT/'manual_review_findings.md').exists())
check('file_exists::manual_review_case_examples.md',     (MAN_OUT/'manual_review_case_examples.md').exists())
check('cases_row_count_10_to_20',                        10 <= len(cases) <= 20, str(len(cases)))
check('case_ids_unique',                                 cases['case_id'].nunique()==len(cases))
check('manual_labels_valid',                             cases['manual_label'].isin(
    ['smooth_success','minor_friction','moderate_friction','severe_friction_or_failure','insufficient_evidence']).all())
check('insufficient_evidence_scores_blank',              cases[cases['manual_label']=='insufficient_evidence']['manual_ground_truth_score'].isna().all())
check('valid_cases_have_scores',                         cases[cases['manual_label']!='insufficient_evidence']['manual_ground_truth_score'].notna().all())

# Recompute MAE and compare
mae_check = True
for m, err_col in metric_map.items():
    expected = round(float(valid[err_col].mean()), 2)
    actual   = float(summary[summary['metric']==m]['mean_absolute_error'].iloc[0])
    if abs(expected - actual) > 0.01:
        mae_check = False
check('summary_mae_recomputed_match', mae_check)

all_pass = all('PASS' in l for l in qc_lines if l.startswith(('PASS','FAIL')))
qc_lines.append('')
qc_lines.append(f'Overall status: {"PASS" if all_pass else "FAIL"}')
(MAN_OUT / 'manual_review_quality_check.md').write_text('\n'.join(qc_lines)+'\n', encoding='utf-8')
print('\n'.join(qc_lines))

# Manual Review Output Quality Check
PASS file_exists::manual_review_cases.csv
PASS file_exists::manual_review_scoring_rubric.csv
PASS file_exists::manual_review_validation_summary.csv
PASS file_exists::manual_review_findings.md
PASS file_exists::manual_review_case_examples.md
PASS cases_row_count_10_to_20: 18
PASS case_ids_unique
PASS manual_labels_valid
PASS insufficient_evidence_scores_blank
PASS valid_cases_have_scores
PASS summary_mae_recomputed_match

Overall status: PASS
